# ML Demand Forecasting with GBT

Generates configurable demand forecasts by store and product using a single
distributed Gradient-Boosted Trees model (Spark ML), tracked with Fabric-native
MLflow.

## Model contract
- Sales are densified to one row per store, product, and calendar day; no-sale
  days carry zero demand.
- Train, calibration, and test windows are chronological rolling origins.
- Every multi-day forecast recursively feeds prior predictions into later lag
  and rolling features.
- Prediction intervals use empirical absolute residual quantiles from the
  calibration origin.
- `mape` is stored as a unit ratio, not percentage points; values can exceed
  `1.0` when error exceeds 100%. `source_as_of` preserves the single maximum
  source timestamp, while `generated_at` records publication time.

The output table name remains configurable through `DEMAND_FORECAST_TABLE`.


In [ ]:
from datetime import timedelta
from functools import reduce
import os

import mlflow
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="demand_forecast")
RECEIPT_LINES_TABLE = get_env(
    "RECEIPT_LINES_TABLE", default="fact_receipt_lines"
)
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
DEMAND_FORECAST_TABLE = get_env(
    "DEMAND_FORECAST_TABLE", default="demand_forecast"
)
DEMAND_FORECAST_TABLE_NAME = (
    f"{LAKEHOUSE_NAME}.{GOLD_DB}.{DEMAND_FORECAST_TABLE}"
)

FORECAST_HORIZON_DAYS = int(get_env("FORECAST_HORIZON_DAYS", default="14"))
MIN_HISTORY_DAYS = int(get_env("MIN_HISTORY_DAYS", default="30"))
MIN_DAILY_SALES = float(get_env("MIN_DAILY_SALES", default="0.5"))
TARGET_MAPE = float(get_env("TARGET_MAPE", default="0.25"))
TARGET_COMBO_PCT = float(get_env("TARGET_COMBO_PCT", default="80.0"))
FORECAST_INTERVAL_COVERAGE = float(
    get_env("FORECAST_INTERVAL_COVERAGE", default="0.95")
)

LAG_DAYS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 14, 28]
MIN_REQUIRED_HISTORY_DAYS = max(
    MIN_HISTORY_DAYS,
    max(LAG_DAYS) + (2 * FORECAST_HORIZON_DAYS) + 1,
)

MAX_ITER = 100
MAX_DEPTH = 6
STEP_SIZE = 0.1
SUBSAMPLING_RATE = 0.8
RANDOM_SEED = 42

if FORECAST_HORIZON_DAYS < 1:
    raise ValueError("FORECAST_HORIZON_DAYS must be positive.")
if not 0.0 < TARGET_MAPE <= 1.0:
    raise ValueError("TARGET_MAPE must be a ratio in the interval (0, 1].")
if not 0.0 < FORECAST_INTERVAL_COVERAGE < 1.0:
    raise ValueError("FORECAST_INTERVAL_COVERAGE must be between 0 and 1.")

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"Output table: {DEMAND_FORECAST_TABLE_NAME}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(
    f"Target MAPE < {TARGET_MAPE:.1%} for "
    f"{TARGET_COMBO_PCT:.1f}% of evaluated combinations"
)
print(
    f"GBT: {MAX_ITER} trees, depth={MAX_DEPTH}, step={STEP_SIZE}, "
    f"seed={RANDOM_SEED}"
)

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_receipt_lines', 'fact_receipts')
ML_OUTPUT_CONTRACTS = {'demand_forecast': [('store_id', 'long', False),
                     ('product_id', 'long', False),
                     ('forecast_date', 'date', False),
                     ('predicted_units', 'double', False),
                     ('lower_bound', 'double', False),
                     ('upper_bound', 'double', False),
                     ('mape', 'double', False),
                     ('source_as_of', 'timestamp', False),
                     ('generated_at', 'timestamp', False),
                     ('forecast_horizon_days', 'int', False),
                     ('model_run_id', 'string', False),
                     ('schema_version', 'string', False)]}


In [ ]:
# =============================================================================
# HELPERS & MLFLOW SETUP
# =============================================================================


def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")


def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")


def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False


def rolling_origin_boundaries(source_as_of_date, horizon_days):
    """Return non-overlapping train, calibration, and test date boundaries."""
    if horizon_days < 1:
        raise ValueError("horizon_days must be positive.")

    test_end = source_as_of_date
    test_start = test_end - timedelta(days=horizon_days - 1)
    calibration_end = test_start - timedelta(days=1)
    calibration_start = calibration_end - timedelta(days=horizon_days - 1)
    train_end = calibration_start - timedelta(days=1)
    return {
        "train_end": train_end,
        "calibration_start": calibration_start,
        "calibration_end": calibration_end,
        "test_start": test_start,
        "test_end": test_end,
    }


def assert_unique_keys(frame, key_columns, context):
    duplicates = (
        frame.groupBy(*key_columns)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    )
    if duplicates:
        raise RuntimeError(f"{context} is not unique by {key_columns}.")


def build_dense_daily_calendar(
    daily_sales,
    calendar_end_date,
    eligibility_end_date,
):
    """Freeze eligibility at a cutoff, then densify through a later date."""
    if eligibility_end_date > calendar_end_date:
        raise ValueError("Eligibility cutoff cannot follow the calendar end.")
    eligibility_sales = daily_sales.filter(
        F.col("sale_date") <= F.lit(eligibility_end_date)
    )
    combo_spans = (
        eligibility_sales.groupBy("store_id", "product_id")
        .agg(F.min("sale_date").alias("first_sale"))
        .withColumn(
            "history_days",
            F.datediff(F.lit(eligibility_end_date), F.col("first_sale")) + 1,
        )
        .filter(F.col("history_days") >= MIN_REQUIRED_HISTORY_DAYS)
    )

    calendar = combo_spans.select(
        "store_id",
        "product_id",
        F.explode(
            F.sequence(F.col("first_sale"), F.lit(calendar_end_date))
        ).alias("sale_date"),
    )
    dense_sales = (
        calendar.join(
            daily_sales,
            on=["store_id", "product_id", "sale_date"],
            how="left",
        )
        .fillna(0.0, subset=["units_sold", "revenue"])
    )
    qualifying = (
        dense_sales.filter(
            F.col("sale_date") <= F.lit(eligibility_end_date)
        ).groupBy("store_id", "product_id")
        .agg(
            F.min("sale_date").alias("first_sale"),
            F.max("sale_date").alias("last_sale"),
            F.count("sale_date").alias("history_days"),
            F.avg("units_sold").alias("avg_daily_sales"),
        )
        .filter(F.col("avg_daily_sales") >= MIN_DAILY_SALES)
    )
    return (
        dense_sales.join(
            qualifying.select("store_id", "product_id"),
            on=["store_id", "product_id"],
            how="inner",
        ),
        qualifying,
    )


def add_demand_features(daily_sales):
    """Add trailing-only lag, rolling, and calendar features."""
    window = Window.partitionBy("store_id", "product_id").orderBy("sale_date")
    features = daily_sales
    for lag_days in LAG_DAYS:
        features = features.withColumn(
            f"lag_{lag_days}d",
            F.lag("units_sold", lag_days).over(window),
        )
    for window_days in ROLLING_WINDOWS:
        features = features.withColumn(
            f"rolling_avg_{window_days}d",
            F.avg("units_sold").over(
                window.rowsBetween(-window_days, -1)
            ),
        )
    return (
        features.withColumn(
            "rolling_std_14d",
            F.stddev("units_sold").over(window.rowsBetween(-14, -1)),
        )
        .withColumn("day_of_week", F.dayofweek("sale_date"))
        .withColumn("day_of_month", F.dayofmonth("sale_date"))
        .withColumn("month", F.month("sale_date"))
        .withColumn(
            "is_weekend",
            F.when(F.dayofweek("sale_date").isin(1, 7), 1).otherwise(0),
        )
        .withColumn("week_of_year", F.weekofyear("sale_date"))
    )


def mape_ratio_column(actual_column, prediction_column):
    """Return absolute percentage error as a 0-1 ratio for positive actuals."""
    return F.when(
        actual_column > 0,
        F.abs(actual_column - prediction_column) / actual_column,
    )


ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



## Data Preparation

In [ ]:
if not silver_exists(RECEIPT_LINES_TABLE) or not silver_exists(RECEIPTS_TABLE):
    raise RuntimeError(
        f"Required tables {RECEIPT_LINES_TABLE} and {RECEIPTS_TABLE} "
        "not found in Silver layer"
    )

print(f"Reading {RECEIPT_LINES_TABLE} and {RECEIPTS_TABLE}...")
df_lines = read_silver(RECEIPT_LINES_TABLE)
df_receipts = read_silver(RECEIPTS_TABLE)

source_as_of_timestamp = (
    df_receipts.select(
        F.max(F.col("event_ts").cast("timestamp")).alias("source_as_of")
    ).first()["source_as_of"]
)
if source_as_of_timestamp is None:
    raise RuntimeError(f"No source timestamps found in {RECEIPTS_TABLE}.")
source_as_of_date = source_as_of_timestamp.date()
origin_dates = rolling_origin_boundaries(
    source_as_of_date,
    FORECAST_HORIZON_DAYS,
)
train_end_date = origin_dates["train_end"]
calibration_start_date = origin_dates["calibration_start"]
calibration_end_date = origin_dates["calibration_end"]
test_start_date = origin_dates["test_start"]
test_end_date = origin_dates["test_end"]

receipt_sales = df_receipts.select(
    "receipt_id_ext",
    "store_id",
    F.col("event_ts").cast("timestamp").alias("sale_ts"),
)
line_sales = df_lines.select(
    "receipt_id_ext",
    "product_id",
    F.col("quantity").cast("double").alias("quantity"),
    F.col("ext_price").cast("double").alias("ext_price"),
)

df_daily_sales = (
    line_sales.join(receipt_sales, on="receipt_id_ext", how="inner")
    .filter(F.col("sale_ts") <= F.lit(source_as_of_timestamp))
    .withColumn("sale_date", F.to_date("sale_ts"))
    .groupBy("store_id", "product_id", "sale_date")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("ext_price").alias("revenue"),
    )
)

raw_daily_rows = df_daily_sales.count()
df_evaluation_base, df_evaluation_cohort = build_dense_daily_calendar(
    df_daily_sales,
    source_as_of_date,
    train_end_date,
)
df_inference_base, df_inference_cohort = build_dense_daily_calendar(
    df_daily_sales,
    source_as_of_date,
    source_as_of_date,
)

evaluation_combinations = df_evaluation_cohort.count()
inference_combinations = df_inference_cohort.count()
if evaluation_combinations == 0:
    raise RuntimeError(
        "No store/product combinations qualify using training-only history."
    )
if inference_combinations == 0:
    raise RuntimeError("No store/product combinations qualify for inference.")

print(f"Global source as-of: {source_as_of_timestamp}")
print(f"Observed non-zero daily rows: {raw_daily_rows}")
print(f"Evaluation dense rows: {df_evaluation_base.count()}")
print(
    f"Training-frozen evaluation combinations: {evaluation_combinations}; "
    f"production inference combinations: {inference_combinations}"
)


## Feature Engineering

Build lag features, rolling averages, and calendar features from daily sales
data. All operations use Spark window functions — fully distributed.

In [ ]:
print("Building trailing demand features...")

df_features = (
    add_demand_features(df_evaluation_base)
    .filter(F.col(f"lag_{max(LAG_DAYS)}d").isNotNull())
    .na.fill(0.0)
)

feature_cols = (
    [f"lag_{days}d" for days in LAG_DAYS]
    + [f"rolling_avg_{days}d" for days in ROLLING_WINDOWS]
    + [
        "rolling_std_14d",
        "day_of_week",
        "day_of_month",
        "month",
        "is_weekend",
        "week_of_year",
        "store_id",
        "product_id",
    ]
)

to_dense_vector = F.udf(
    lambda *values: Vectors.dense(
        [0.0 if value is None else float(value) for value in values]
    ),
    VectorUDT(),
)


def with_feature_vector(frame):
    """Cast model inputs and assemble a deterministic dense feature vector."""
    cleaned = frame
    for column_name in feature_cols:
        cleaned = cleaned.withColumn(
            column_name,
            F.col(column_name).cast("double"),
        )
    if "units_sold" in cleaned.columns:
        cleaned = cleaned.withColumn(
            "units_sold", F.col("units_sold").cast("double")
        )
    return cleaned.withColumn(
        "features",
        to_dense_vector(*[F.col(column_name) for column_name in feature_cols]),
    )


def forecast_recursively(model, observed_history, forecast_start, horizon_days):
    """Forecast a horizon while appending each prediction to feature state."""
    state = (
        observed_history.select(
            F.col("store_id").cast("double").alias("store_id"),
            F.col("product_id").cast("double").alias("product_id"),
            F.col("sale_date").cast("date").alias("sale_date"),
            F.col("units_sold").cast("double").alias("units_sold"),
        )
        .filter(F.col("sale_date") < F.lit(forecast_start))
    )
    combinations = state.select("store_id", "product_id").distinct().cache()
    combination_count = combinations.count()
    if combination_count == 0:
        raise RuntimeError("Recursive forecast has no store/product history.")

    forecast_frames = []
    for horizon_step in range(1, horizon_days + 1):
        forecast_date = forecast_start + timedelta(days=horizon_step - 1)
        placeholder = combinations.select(
            "store_id",
            "product_id",
            F.lit(forecast_date).cast("date").alias("sale_date"),
            F.lit(None).cast("double").alias("units_sold"),
        )
        candidate_features = (
            add_demand_features(state.unionByName(placeholder))
            .filter(F.col("sale_date") == F.lit(forecast_date))
            .na.fill(0.0)
        )
        step_predictions = (
            model.transform(with_feature_vector(candidate_features))
            .withColumn(
                "predicted_units",
                F.greatest(F.col("predicted_units"), F.lit(0.0)),
            )
            .select(
                "store_id",
                "product_id",
                F.col("sale_date").alias("forecast_date"),
                "predicted_units",
                F.lit(horizon_step).alias("horizon_step"),
            )
            .cache()
        )
        if step_predictions.count() != combination_count:
            raise RuntimeError(
                f"Recursive horizon step {horizon_step} lost store/product rows."
            )
        forecast_frames.append(step_predictions)
        state = state.unionByName(
            step_predictions.select(
                "store_id",
                "product_id",
                F.col("forecast_date").alias("sale_date"),
                F.col("predicted_units").alias("units_sold"),
            )
        )

    return reduce(lambda left, right: left.unionByName(right), forecast_frames)


def calibration_residual_quantiles(
    calibration_predictions,
    calibration_actuals,
    coverage,
):
    """Estimate empirical absolute-residual quantiles by horizon step."""
    residuals = (
        calibration_predictions.join(
            calibration_actuals,
            on=["store_id", "product_id", "forecast_date"],
            how="inner",
        )
        .withColumn(
            "absolute_residual",
            F.abs(F.col("units_sold") - F.col("predicted_units")),
        )
    )
    quantiles = residuals.groupBy("horizon_step").agg(
        F.percentile_approx(
            "absolute_residual",
            coverage,
            10000,
        ).alias("residual_quantile")
    )
    if quantiles.count() != FORECAST_HORIZON_DAYS:
        raise RuntimeError(
            "Calibration did not produce a residual quantile for every horizon."
        )
    return quantiles


print(f"Features ({len(feature_cols)}): {', '.join(feature_cols)}")
print(f"Feature rows after lag trimming: {df_features.count()}")


In [ ]:
df_train = df_features.filter(F.col("sale_date") <= F.lit(train_end_date))
df_calibration_actual = df_evaluation_base.filter(
    (F.col("sale_date") >= F.lit(calibration_start_date))
    & (F.col("sale_date") <= F.lit(calibration_end_date))
)
df_test_actual = df_evaluation_base.filter(
    (F.col("sale_date") >= F.lit(test_start_date))
    & (F.col("sale_date") <= F.lit(test_end_date))
)

split_counts = {
    "train": df_train.count(),
    "calibration": df_calibration_actual.count(),
    "test": df_test_actual.count(),
}
if any(count == 0 for count in split_counts.values()):
    raise RuntimeError(
        f"Rolling-origin split produced an empty window: {split_counts}. "
        "Increase source history or reduce FORECAST_HORIZON_DAYS."
    )

print(f"Global source as-of date: {source_as_of_date}")
print(f"Train through: {train_end_date} ({split_counts['train']} rows)")
print(
    f"Calibration: {calibration_start_date} to {calibration_end_date} "
    f"({split_counts['calibration']} rows)"
)
print(
    f"Test: {test_start_date} to {test_end_date} "
    f"({split_counts['test']} rows)"
)


## Model Training & Evaluation

In [ ]:
df_train_assembled = with_feature_vector(df_train)

print("Training Spark GBT demand model...")
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="units_sold",
    predictionCol="predicted_units",
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=RANDOM_SEED,
)

with mlflow.start_run(
    run_name=f"gbt_demand_{source_as_of_timestamp.strftime('%Y%m%d_%H%M')}"
) as run:
    model = gbt.fit(df_train_assembled.select("features", "units_sold"))

    calibration_predictions = forecast_recursively(
        model,
        df_evaluation_base.filter(F.col("sale_date") <= F.lit(train_end_date)),
        calibration_start_date,
        FORECAST_HORIZON_DAYS,
    )
    calibration_actuals = df_calibration_actual.select(
        "store_id",
        "product_id",
        F.col("sale_date").alias("forecast_date"),
        F.col("units_sold").cast("double").alias("units_sold"),
    )
    residual_quantiles = calibration_residual_quantiles(
        calibration_predictions,
        calibration_actuals,
        FORECAST_INTERVAL_COVERAGE,
    ).cache()

    test_predictions = forecast_recursively(
        model,
        df_evaluation_base.filter(
            F.col("sale_date") <= F.lit(calibration_end_date)
        ),
        test_start_date,
        FORECAST_HORIZON_DAYS,
    )
    test_actuals = df_test_actual.select(
        "store_id",
        "product_id",
        F.col("sale_date").alias("forecast_date"),
        F.col("units_sold").cast("double").alias("units_sold"),
    )
    df_test_evaluation = test_predictions.join(
        test_actuals,
        on=["store_id", "product_id", "forecast_date"],
        how="inner",
    ).withColumn(
        "absolute_percentage_error",
        mape_ratio_column(F.col("units_sold"), F.col("predicted_units")),
    )

    overall_mape = (
        df_test_evaluation.agg(
            F.avg("absolute_percentage_error").alias("mape")
        ).first()["mape"]
        or 0.0
    )
    mape_by_combo = df_test_evaluation.groupBy(
        "store_id", "product_id"
    ).agg(F.avg("absolute_percentage_error").alias("avg_mape"))
    stats = mape_by_combo.agg(
        F.avg("avg_mape").alias("mean_mape"),
        F.percentile_approx("avg_mape", 0.5, 10000).alias("median_mape"),
        F.percentile_approx("avg_mape", 0.9, 10000).alias("p90_mape"),
    ).first()
    mean_combo_mape = float(stats["mean_mape"] or 0.0)
    median_combo_mape = float(stats["median_mape"] or 0.0)
    p90_combo_mape = float(stats["p90_mape"] or 0.0)

    evaluated_combinations = mape_by_combo.count()
    meeting_target = mape_by_combo.filter(
        F.col("avg_mape") < TARGET_MAPE
    ).count()
    pct_meeting = (
        meeting_target / evaluated_combinations * 100.0
        if evaluated_combinations
        else 0.0
    )
    rmse = (
        df_test_evaluation.agg(
            F.sqrt(
                F.avg(
                    F.pow(F.col("units_sold") - F.col("predicted_units"), 2)
                )
            ).alias("rmse")
        ).first()["rmse"]
        or 0.0
    )

    mlflow.log_params({
        "algorithm": "spark_gbt_regressor",
        "forecast_horizon_days": FORECAST_HORIZON_DAYS,
        "interval_coverage": FORECAST_INTERVAL_COVERAGE,
        "min_history_days": MIN_REQUIRED_HISTORY_DAYS,
        "min_daily_sales": MIN_DAILY_SALES,
        "target_mape_ratio": TARGET_MAPE,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "evaluation_combinations": evaluation_combinations,
        "inference_combinations": inference_combinations,
        "train_rows": split_counts["train"],
        "calibration_rows": split_counts["calibration"],
        "test_rows": split_counts["test"],
        "source_as_of": source_as_of_timestamp.isoformat(),
    })
    mlflow.log_metrics({
        "overall_mape_ratio": round(float(overall_mape), 6),
        "mean_combo_mape_ratio": round(mean_combo_mape, 6),
        "median_combo_mape_ratio": round(median_combo_mape, 6),
        "p90_combo_mape_ratio": round(p90_combo_mape, 6),
        "rmse": round(float(rmse), 4),
        "pct_meeting_target": round(pct_meeting, 1),
        "evaluated_combinations": evaluated_combinations,
    })
    mlflow.spark.log_model(model, "gbt_demand_model")

    print(f"MLflow run: {run.info.run_id}")
    print(f"Recursive test MAPE: {overall_mape:.1%}")
    print(f"Recursive test RMSE: {rmse:.2f}")
    print(
        "Per-combination MAPE -- "
        f"mean: {mean_combo_mape:.1%}, "
        f"median: {median_combo_mape:.1%}, "
        f"p90: {p90_combo_mape:.1%}"
    )
    print(
        f"Meeting target (<{TARGET_MAPE:.1%}): {meeting_target} "
        f"({pct_meeting:.1f}%)"
    )


## Generate Forecasts

Use the trained model to predict the next 14 days. For multi-step forecasting,
we use the most recent actual data as the starting features for each
store x product combination.

In [ ]:
print(
    f"Generating a recursive {FORECAST_HORIZON_DAYS}-day forecast from "
    f"the global source as-of {source_as_of_timestamp}..."
)

forecast_start_date = source_as_of_date + timedelta(days=1)
df_all_forecasts = forecast_recursively(
    model,
    df_inference_base,
    forecast_start_date,
    FORECAST_HORIZON_DAYS,
)

df_all_forecasts = (
    df_all_forecasts.join(
        residual_quantiles,
        on="horizon_step",
        how="inner",
    )
    .withColumn(
        "lower_bound",
        F.greatest(
            F.col("predicted_units") - F.col("residual_quantile"),
            F.lit(0.0),
        ),
    )
    .withColumn(
        "upper_bound",
        F.col("predicted_units") + F.col("residual_quantile"),
    )
    .withColumn("mape", F.lit(float(overall_mape)))
    .withColumn(
        "source_as_of",
        F.lit(source_as_of_timestamp).cast("timestamp"),
    )
    .withColumn(
        "generated_at",
        F.current_timestamp(),
    )
)

df_final = df_all_forecasts.select(
    F.col("store_id").cast("long"),
    F.col("product_id").cast("long"),
    F.col("forecast_date").cast("date"),
    F.col("predicted_units").cast("double"),
    F.col("lower_bound").cast("double"),
    F.col("upper_bound").cast("double"),
    F.col("mape").cast("double"),
    F.col("source_as_of").cast("timestamp"),
    F.col("generated_at").cast("timestamp"),
    F.lit(FORECAST_HORIZON_DAYS).cast("int").alias("forecast_horizon_days"),
    F.lit(run.info.run_id).cast("string").alias("model_run_id"),
    F.lit("1.0").cast("string").alias("schema_version"),
)

assert_unique_keys(
    df_final,
    ["store_id", "product_id", "forecast_date"],
    "Demand forecast output",
)

df_final = validate_ml_output(df_final, "demand_forecast")
df_final.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(
    DEMAND_FORECAST_TABLE_NAME
)

forecast_count = spark.table(DEMAND_FORECAST_TABLE_NAME).count()
combinations_forecasted = (
    spark.table(DEMAND_FORECAST_TABLE_NAME)
    .select("store_id", "product_id")
    .distinct()
    .count()
)
print(f"Saved {forecast_count} forecast rows to {DEMAND_FORECAST_TABLE_NAME}")
print(f"Covering {combinations_forecasted} store/product combinations")


In [ ]:
print("=" * 60)
print("DEMAND FORECASTING COMPLETE")
print("=" * 60)

gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold layer ({GOLD_DB}): {len(gold_tables)} tables")
print(f"Forecast table: {DEMAND_FORECAST_TABLE_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"\nMLflow experiment: {EXPERIMENT_NAME}")
print(f"Model logged: gbt_demand_model")
print("View results in Fabric > Experiments")
print("\nSchedule this notebook on your preferred cadence for fresh forecasts.")